# Rossmann Sales Forecasting - Model Training

### The goal of this notebook is to test Machine Learning models predict daily sales across Rossmann stores.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Define project paths
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "processed").is_dir()
)

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# Load feature-engineered datasets
train_df = pd.read_csv(
    PROCESSED_PATH / "train_features.csv",
    parse_dates=["date"],
    dtype={"state_holiday": "string"}
)

test_df = pd.read_csv(
    PROCESSED_PATH / "test_features.csv",
    parse_dates=["date"],
    dtype={"state_holiday": "string"}
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (1017209, 26)
Test: (41088, 25)


### 2. Temporal Train/Validation Split

In [4]:
# Define temporal cutoff
cutoff_date = train_df["date"].max() - pd.Timedelta(weeks=6)

# Chronological split
train_data = train_df[
    train_df["date"] < cutoff_date
].copy()

val_data = train_df[
    train_df["date"] >= cutoff_date
].copy()

# Validate split
print("Training set:", train_data.shape)
print("Validation set:", val_data.shape)

print("\nTraining period:")
print(train_data["date"].min(), "to", train_data["date"].max())

print("\nValidation period:")
print(val_data["date"].min(), "to", val_data["date"].max())

Training set: (969264, 26)
Validation set: (47945, 26)

Training period:
2013-01-01 00:00:00 to 2015-06-18 00:00:00

Validation period:
2015-06-19 00:00:00 to 2015-07-31 00:00:00


**Observation:** A chronological train-validation split was applied to simulate real-world sales forecasting. The most recent six weeks were reserved for validation, allowing the models to be evaluated on observations occurring after the training period.

Unlike a random split, this approach preserves the temporal order of the data and reduces the risk of data leakage.

### 3. Defining Features and Target

In [5]:
# Define target variable
target = "sales"

# Columns excluded from model inputs
exclude_cols = ["sales", "customers", "date", "store"]

# Define features and target
X_train = train_data.drop(columns=exclude_cols).copy()
y_train = train_data[target].copy()

X_val = val_data.drop(columns=exclude_cols).copy()
y_val = val_data[target].copy()

# Preserve store IDs for later analysis
store_train = train_data["store"].copy()
store_val = val_data["store"].copy()

# Validate dimensions
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nSame features:", X_train.columns.equals(X_val.columns))


X_train: (969264, 22)
y_train: (969264,)

X_val: (47945, 22)
y_val: (47945,)

Same features: True


**Observation:** The dataset was separated into 22 predictor variables and the target variable (`sales`). Store IDs were preserved for future store-level analysis but excluded from the initial model, along with customer counts to prevent data leakage.

Both training and validation sets contain the same features, ensuring consistency during modelling.

In [6]:
# Identify numerical and categorical features
numerical_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['day_of_week', 'open', 'promo', 'school_holiday', 'competition_distance', 'competition_open_since_month', 'competition_open_since_year', 'promo2', 'promo2_since_week', 'promo2_since_year', 'year', 'month', 'day_of_month', 'week_of_year', 'promo2_weeks_since_start', 'promo2_active', 'competition_months_open', 'competition_start_known']

Categorical features: ['state_holiday', 'store_type', 'assortment', 'promo_interval']


In [7]:
# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [8]:
# Treat calendar categories appropriately
calendar_categorical = ["day_of_week", "month"]

numerical_features = [
    col for col in numerical_features
    if col not in calendar_categorical
]

categorical_features += calendar_categorical

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Numerical features: 16
Categorical features: 6


**Observation:** The dataset contains 22 predictor variables. Although some calendar features are stored as integers, `day_of_week` and `month` were treated as categorical to avoid introducing artificial linear relationships in the initial regression model.

Numerical and categorical features will undergo separate preprocessing transformations, fitted exclusively on the training data.

In [9]:
# Calculate average sales on open days (training data only)
avg_open_sales = y_train[X_train["open"] == 1].mean()

# Generate baseline predictions
baseline_pred = np.where(
    X_val["open"] == 1,
    avg_open_sales,
    0
)

print(f"Average sales on open days: €{avg_open_sales:,.2f}")

Average sales on open days: €6,953.47


In [10]:
# Calculate evaluation metrics
baseline_mae = mean_absolute_error(y_val, baseline_pred)

baseline_rmse = np.sqrt(
    mean_squared_error(y_val, baseline_pred)
)

# Display results
baseline_results = pd.DataFrame({
    "Model": ["Baseline - Average Sales"],
    "MAE": [baseline_mae],
    "RMSE": [baseline_rmse]
})

display(baseline_results.round(2))

,Model,MAE,RMSE
0,Baseline - Average Sales,1935.84,2828.75


**Observation:** A baseline forecasting model was developed using historical average sales for open stores and zero sales for closed stores.

This simple approach provides a reference for evaluating more advanced regression models. Model performance will be compared using MAE and RMSE, where lower values indicate better predictions.

The baseline model achieved an MAE of €1,935.84 and an RMSE of €2,828.75 on the validation set.

These results provide a benchmark for evaluating more advanced models. Since the baseline uses the same historical average for all open stores, it does not capture store-specific characteristics, promotional effects, or seasonal patterns.

### 6. Linear Regression Model

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Build Linear Regression pipeline
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train using open stores only
open_train_mask = X_train["open"] == 1

linear_model.fit(
    X_train.loc[open_train_mask],
    y_train.loc[open_train_mask]
)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
# Initialize predictions
linear_pred = np.zeros(len(X_val))

# Predict sales for open stores
open_val_mask = X_val["open"] == 1

linear_pred[open_val_mask.to_numpy()] = (
    linear_model.predict(X_val.loc[open_val_mask])
)

# Avoid negative sales predictions
linear_pred = np.maximum(linear_pred, 0)

In [13]:
# Calculate evaluation metrics
linear_mae = mean_absolute_error(y_val, linear_pred)

linear_rmse = np.sqrt(
    mean_squared_error(y_val, linear_pred)
)

# Compare with baseline
model_comparison = pd.DataFrame({
    "Model": [
        "Baseline - Average Sales",
        "Linear Regression"
    ],
    "MAE": [
        baseline_mae,
        linear_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse
    ]
})

display(model_comparison.round(2))

,Model,MAE,RMSE
0,Baseline - Average Sales,1935.84,2828.75
1,Linear Regression,1643.41,2417.21


**Observation:** Linear Regression outperformed the baseline model, reducing MAE from €1,935.84 to €1,643.41 (15.1%) and RMSE from €2,828.75 to €2,417.21 (14.5%).

These results suggest that incorporating store characteristics, promotional information, and calendar features improves sales forecasting compared to using historical average sales alone.

In [14]:
# Evaluate model performance on open days only
open_mask = X_val["open"] == 1

open_days_results = pd.DataFrame({
    "Model": [
        "Baseline - Average Sales",
        "Linear Regression"
    ],
    "MAE": [
        mean_absolute_error(
            y_val[open_mask],
            baseline_pred[open_mask.to_numpy()]
        ),
        mean_absolute_error(
            y_val[open_mask],
            linear_pred[open_mask.to_numpy()]
        )
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(
            y_val[open_mask],
            baseline_pred[open_mask.to_numpy()]
        )),
        np.sqrt(mean_squared_error(
            y_val[open_mask],
            linear_pred[open_mask.to_numpy()]
        ))
    ]
})

display(open_days_results.round(2))

,Model,MAE,RMSE
0,Baseline - Average Sales,2242.1,3044.3
1,Linear Regression,1903.4,2601.4


**Observation:** When evaluating only open days, Linear Regression achieved an MAE of €1,903.40 and an RMSE of €2,601.40, outperforming the baseline by approximately 15.1% and 14.5%, respectively.

Errors are higher than in the overall evaluation because closed days, where sales are typically zero, are excluded. This provides a more representative assessment of forecasting performance during business operations.